In [ ]:
# =============
# LIBRARY IMPORTS

from astroquery.utils.tap.core import TapPlus
from astroquery.ipac.ned import Ned
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import pandas as pd
import pickle
import re
import requests
import time
from SciServer import Authentication
from SciServer import CasJobs as cj

# Authenticate session
username = "username"
password = "password"
Authentication.login(UserName = username, Password = password)

In [ ]:
# =============
# LOOP TO RETRIEVE SDSS .fits STARTING FROM A MINIMUM REDSHIFT UPDATED EACH ITERATION

# Update minimum redshift
# ultimo_redshift = 0
# with open('extra/redshift_init.pickle', 'wb') as handle:
#     pickle.dump(ultimo_redshift, handle)

# Initial parameters
batch_size_inicial = 50000       # tamaño inicial del lote
min_batch_size = 1000            # tamaño minimo de lote permitido
max_retries = 10                 # maximo numero de reintentos por lote
offset = 0                       # parametro de parada
all_results = []
batch_results = []

while offset < 1000000:

    with open('extra/redshift_init.pickle', 'rb') as handle:
        redshift_init = pickle.load(handle)

    redshift_init = float(str(redshift_init)[:8]) # Solo se toman los primeros 7 digitos, si no se excede el límite de la consulta
    current_batch_size = batch_size_inicial
    retries = 0
    success = False

    # Try executing query with current batch size; reduce and retry on failure
    while not success and retries < max_retries:
        sql_query = f"""
        SELECT
            p.objid,
            s.specObjID,
            dbo.fGetUrlFitsSpectrum(s.specObjID) AS fits_url,
            s.z AS redshift
        FROM 
            PhotoObj AS p
        JOIN 
            SpecObj AS s ON s.bestobjid = p.objid
        WHERE 
            s.z BETWEEN {redshift_init} AND 9
            AND s.zWarning = 0
            AND (s.class = 'QSO' OR s.class = 'GALAXY')
        ORDER BY s.z
        OFFSET 0 ROWS
        FETCH NEXT {current_batch_size} ROWS ONLY
        """
        try:
            batch_results = pd.DataFrame(cj.executeQuery(sql_query, context="DR18"))
            success = True
            all_results.append(batch_results)
            offset += len(batch_results)

            with open('extra/all_results.pickle', 'wb') as handle:
                pickle.dump(all_results, handle)
            print(f"Retrieved {offset} records so far.")

            ultimo_redshift = batch_results["redshift"].iloc[-1]

            ultimo_redshift = float(str(ultimo_redshift)[:8])
            print(f"Initial redshift: {redshift_init}")
            print(f"Last redshift: {ultimo_redshift}")
            if ultimo_redshift == redshift_init:    # Si el redshift no cambio, se incrementa un poco para evitar loops infinitos
                ultimo_redshift = ultimo_redshift + 0.000002
                print(f"Increased redshift to {ultimo_redshift}")

            with open('extra/redshift_init.pickle', 'wb') as handle:
                pickle.dump(ultimo_redshift, handle)

        except Exception as e:
            print(f"Query error with batch_size = {current_batch_size}: {e}")
            # Reduce batch size without dropping below min_batch_size
            current_batch_size = max(min_batch_size, current_batch_size // 2)
            retries += 1
            time.sleep(2)  # Esperar un poco antes de reintentar

    if len(batch_results) == 0:
        print("No more records to retrieve.")
        break

    if retries == max_retries:
        print("Could not retrieve batch after multiple retries. Aborting.")
        break

In [ ]:
with open('extra/all_results.pickle', 'rb') as handle:
    all_results = pickle.load(handle)

# Combine all results
combined_results = pd.concat(all_results, ignore_index=True)
print(f"Total records retrieved: {len(combined_results)}")

# Extract FITS file URLs
fits_urls = combined_results["fits_url"].tolist()
print(f"Retrieved {len(fits_urls)} FITS URLs.")

ultimo_redshift = all_results[len(all_results)-2]["redshift"].iloc[0]
print(f"Last retrieved redshift: {ultimo_redshift:.7f}")

with open('extra/redshift_init.pickle', 'wb') as handle:
    pickle.dump(ultimo_redshift, handle)

In [ ]:
# =============
# PARALLEL DOWNLOAD OF RETRIEVED .fits FILES

# Download directory
output_dir = r"spectrums"
os.makedirs(output_dir, exist_ok=True)

archivos_descargados = []

def download_file(url):
    filename = os.path.basename(url)
    local_path = os.path.join(output_dir, filename)
    
    # Skip if file is already downloaded
    if os.path.exists(local_path):
        print(f"Skipping {filename} (already downloaded).")
        return filename, None
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(response.content)
        print(f"Downloaded: {filename}")
        return filename, None
    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return filename, e

# Number of parallel downloads
paralelo = 100
i=0

with ThreadPoolExecutor(max_workers=paralelo) as executor:
    future_to_url = {executor.submit(download_file, url): url for url in fits_urls}
    
    for future in as_completed(future_to_url):
        i+=1
        print(f"Downloading file {i} of {len(fits_urls)}")
        filename, error = future.result()
        if error is None:
            archivos_descargados.append(filename)

print("Download complete.")

In [ ]:
# =============
# COMPACT CODE FOR DOWNLOADING NED SPECTRA

# Functions and initial configuration

def sanitize(name):
    """Reemplaza caracteres inválidos para Windows por '_'."""
    return re.sub(r'[^A-Za-z0-9_-]', '_', name)

output_dir    = "spectrums NASA"
redshift_path = 'extra/redshift_init_ned.pickle'
os.makedirs(output_dir, exist_ok=True)

# Update minimum redshift
redshift_init = 0
with open(redshift_path, 'wb') as handle:
    pickle.dump(redshift_init, handle)

batch_size_in = 10000      # lote inicial ADQL
max_retries   = 5          # reintentos por lote

all_meta = []              # acumulador de metadatos

# Redshift pagination for metadata
while True:

    with open(redshift_path, 'rb') as f:
        redshift_init = float(str(pickle.load(f))[:8])
        print(f"Current redshift: {redshift_init}")
        print(f"Current redshift: {redshift_init:.7f}")

    current_batch = batch_size_in
    retries       = 0
    success       = False

    # Query attempts with batch reduction
    while not success and retries < max_retries:
        adql = f"""
        SELECT TOP {current_batch}
            prefname, z
        FROM NEDTAP.objdir
        WHERE
            z >= {redshift_init:.7f}
            AND z < 9.0
            AND n_spectra > 0
        ORDER BY z
        """
        try:
            tap     = TapPlus(url="https://ned.ipac.caltech.edu/tap")
            batch   = tap.launch_job(adql).get_results().to_pandas()
            success = True

            if batch.empty:
                break

            all_meta.append(batch)

            # Update z_init
            last_z = float(str(batch['z'].iloc[-1])[:8])
            if last_z == redshift_init:
                last_z += 0.000002
            with open(redshift_path, 'wb') as f:
                pickle.dump(last_z, f)

        except Exception as e:
            retries += 1
            print(e)
            time.sleep(2)

    if not success or batch.empty:
        break

meta_df = pd.concat(all_meta, ignore_index=True)

# Parallel download handling existing files
def download_spectra(prefname, z):
    base = sanitize(prefname)
    saved = []
    try:
        spectra = Ned.get_spectra(prefname, show_progress=False)
    except Exception:
        return saved

    for idx, hdulist in enumerate(spectra, start=1):
        filename = f"{base}_{idx}_z{z:.5f}.fits"
        path     = os.path.join(output_dir, filename)
        if os.path.exists(path):
            print(f"Skipping (already exists): {filename}")
        else:
            hdulist.writeto(path, overwrite=True)
            print(f"Saved: {filename}")
        saved.append(path)
    return saved

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {
        executor.submit(download_spectra, row['prefname'], row['z']): ix
        for ix, row in meta_df.iterrows()
    }
    for future in as_completed(futures):
        pass

print(f"Final redshift: {redshift_init}")